In [17]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

In [18]:
df = pd.read_parquet('features.parquet')

In [63]:
df_hmm = df.copy()
df_hmm["datetime"] = pd.to_datetime(df_hmm["datetime"], utc=True)
df_hmm = df_hmm.sort_values("datetime").set_index("datetime")

r = df_hmm["r"]
close = df_hmm["close"]
log_close = np.log(close)

df_hmm["ret_6"] = r.rolling(6).sum()
df_hmm["ret_24"] = r.rolling(24).sum()
df_hmm["rv_42"] = np.sqrt((r ** 2).rolling(42).sum())
df_hmm["rv_72"] = np.sqrt((r ** 2).rolling(72).sum())

df_hmm["down_rv_72"] = np.sqrt((np.minimum(r, 0.0) ** 2).rolling(72).sum())
df_hmm["up_rv_72"] = np.sqrt((np.maximum(r, 0.0) ** 2).rolling(72).sum())
df_hmm["downside_share_72"] = df_hmm["down_rv_72"] / (
    df_hmm["down_rv_72"] + df_hmm["up_rv_72"]
)
df_hmm["rv_ratio_24_72"] = (
    np.sqrt((r ** 2).rolling(24).sum()) / df_hmm["rv_72"]
)

df_hmm["ma_dist_72"] = log_close - log_close.rolling(72).mean()
df_hmm["ma_slope_42"] = log_close.rolling(42).mean().diff(6)
df_hmm["drawdown_180"] = close / close.rolling(180).max() - 1
df_hmm["ret_24_over_rv_72"] = df_hmm["ret_24"] / df_hmm["rv_72"]

feature_cols = [
    "r",                  
    "roll_skew",        
    "ret_6",             
    "ret_24",            
    "rv_42",             
    "rv_72",              
    "rv_ratio_24_72",    
    "downside_share_72", 
    "ma_dist_72",         
    "ma_slope_42",       
    "drawdown_180",       
    "ret_24_over_rv_72", 
]

df_hmm = (
    df_hmm[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

X = df_hmm[feature_cols].to_numpy(dtype=float)
Y = np.zeros(len(df_hmm), dtype=int)
dates = df_hmm.index

r_col = feature_cols.index("r")

In [5]:
df_hmm.head()

,r,real_vol,range_pct,vol_ratio,mom,roll_skew
datetime,,,,,,
2020-04-02 08:00:00+00:00,0.005482,0.009154,0.015803,0.939951,0.084590,0.678023
2020-04-02 12:00:00+00:00,0.017889,0.009471,0.030530,1.836026,0.104356,0.672250
2020-04-02 16:00:00+00:00,0.000000,0.008056,0.074671,5.040902,0.141964,0.676163
2020-04-02 20:00:00+00:00,0.000000,0.008040,0.031707,1.195707,0.146042,0.722093
2020-04-03 00:00:00+00:00,-0.002283,0.007612,0.014968,0.663369,0.122949,0.746489


In [50]:
def walk_forward(X, Y, dates, model, tr, v, te, em, **model_kwargs):
    preds = []

    n = len(Y)
    t = np.arange(n)

    for k in range(0, n - tr - v - 2 * em - te + 1, te):
        train = t[k : k + tr]
        val = t[k + tr + em : k + tr + em + v]
        test = t[k + tr + 2 * em + v : k + tr + 2 * em + v + te]

        X_train = X[train]
        Y_train = Y[train]

        X_val = X[val]
        Y_val = Y[val]

        X_test = X[test]
        Y_test = Y[test]
        Y_hat = model(
            X_train=X_train,
            Y_train=Y_train,
            X_val=X_val,
            Y_val=Y_val,
            X_test=X_test,
            dates=dates,
            X_all=X,
            train_idx=train,
            val_idx=val,
            test_idx=test,
            **model_kwargs,)

        fold = pd.DataFrame(
            {
                "Y_true": Y_test,
                "Y_pred": Y_hat,
            },
            index=dates[test],)
        preds.append(fold)

    return pd.concat(preds)

In [45]:
def make_future_return_target(X_all, end_idx, horizon=24, r_col=0):
    y = []
    for i in end_idx:
        future_r = X_all[i + 1 : i + 1 + horizon, r_col]
        y.append(np.sum(future_r))
    return np.array(y, dtype=float)

In [46]:
def _normalize(p):
    s = p.sum()
    if s <= 0 or not np.isfinite(s):
        return np.ones_like(p) / len(p)
    return p / s
def _filtered_probs_hmm(model, X_seq, init_alpha=None):
    # log p(x_t | S_t = k), shape = (T, K)
    log_lik = model._compute_log_likelihood(X_seq)
    lik = np.exp(log_lik - log_lik.max(axis=1, keepdims=True))
    T, K = lik.shape
    alpha = np.zeros((T, K))
    if init_alpha is None:
        prev = model.startprob_.copy()
    else:
        prev = init_alpha.copy()
    prev = _normalize(prev)
    for t in range(T):
        if t == 0:
            pred = prev
        else:
            pred = alpha[t - 1] @ model.transmat_
        alpha[t] = _normalize(pred * lik[t])
    return alpha

In [82]:
def hmm_model(
    X_train,
    Y_train,
    dates,
    X_val=None,
    Y_val=None,
    X_test=None,
    **kwargs,
):
    X_all = kwargs["X_all"]
    train_idx = kwargs["train_idx"]
    test_idx = kwargs["test_idx"]

    pred_horizon = kwargs.get("pred_horizon", 24)
    r_col = kwargs.get("r_col", 0)
    n_components = kwargs.get("n_components", 2)

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    hmm = GaussianHMM(
        n_components=n_components,
        covariance_type="diag",
        n_iter=300,
        tol=1e-3,
        random_state=5,
        verbose=False,
        min_covar=1e-4,
        implementation="log",
    )

    hmm.fit(X_train_s)

    train_filt = _filtered_probs_hmm(hmm, X_train_s)

    valid_mask = train_idx <= train_idx[-1] - pred_horizon
    valid_train_idx = train_idx[valid_mask]
    valid_train_filt = train_filt[valid_mask]

    y_fwd_ret = make_future_return_target(
        X_all=X_all,
        end_idx=valid_train_idx,
        horizon=pred_horizon,
        r_col=r_col,
    )

    regime_score = []

    for k in range(hmm.n_components):
        w = valid_train_filt[:, k]
        score = np.sum(w * y_fwd_ret) / np.sum(w)
        regime_score.append(score)

    regime_score = np.array(regime_score)
    order = np.argsort(regime_score)

    init_test = train_filt[-1].copy()

    steps_ahead = test_idx[0] - train_idx[-1]

    for _ in range(steps_ahead):
        init_test = init_test @ hmm.transmat_

    test_filt = _filtered_probs_hmm(
        hmm,
        X_test_s,
        init_alpha=init_test,
    )

    test_filt = test_filt[:, order]

    return [tuple(row) for row in test_filt]

In [95]:
preds_hmm = walk_forward(
    X=X,
    Y=Y,
    dates=dates,
    model=hmm_model,
    tr=5000,
    v=0,
    te=42,
    em=4,
    pred_horizon=24,
    r_col=r_col,
    n_components=5,)

In [96]:
h = 24

res = preds_hmm.copy()
res["regime"] = res["Y_pred"].apply(lambda p: int(np.argmax(p)))
res = res.reset_index().rename(columns={"index": "datetime"})

px = df[["datetime", "close", "r"]].copy()
px["datetime"] = pd.to_datetime(px["datetime"], utc=True)
px = px.sort_values("datetime")

px["fwd_ret"] = px["close"].shift(-h) / px["close"] - 1
px["fwd_log_ret"] = px["r"].rolling(h).sum().shift(-h + 1)

check = res.merge(
    px[["datetime", "r", "fwd_ret", "fwd_log_ret"]],
    on="datetime",
    how="inner",
)

print(
    check.groupby("regime")[["r", "fwd_ret", "fwd_log_ret"]]
         .agg(["count", "mean", "median", "std"])
)

           r                               fwd_ret                      \
       count      mean    median       std   count      mean    median   
regime                                                                   
0       1588 -0.000248  0.000083  0.011439    1588  0.002670  0.002976   
1       2317 -0.000112  0.000011  0.007768    2317  0.003249  0.000288   
2       1908 -0.000198  0.000051  0.008949    1908  0.001424 -0.000517   
3       1274  0.000206  0.000265  0.011476    1274  0.004828  0.004676   
4       1019  0.001903  0.000611  0.011389    1019  0.017059  0.006679   

                 fwd_log_ret                                
             std       count      mean    median       std  
regime                                                      
0       0.048393        1588  0.001532  0.002695  0.047688  
1       0.046737        2317  0.001626 -0.000350  0.046176  
2       0.045889        1908  0.000156 -0.000079  0.045322  
3       0.053329        1274  0.003507  0

In [88]:
proba_hmm = np.vstack(preds_hmm["Y_pred"].to_numpy())

hmm_probs = pd.DataFrame(
    proba_hmm,
    index=preds_hmm.index,
    columns=[
        "p_regime_0",
        "p_regime_1",
        "p_regime_2",
        "p_regime_3",
        "p_regime_4",
    ],
)

hmm_probs.index.name = "datetime"
hmm_probs = hmm_probs.reset_index()

hmm_probs.to_parquet(
    "hmm_regime_probabilities_oos_filtered.parquet",
    index=False,
    engine="pyarrow",
)

In [89]:
hmm_probs.head(50)

,datetime,p_regime_0,p_regime_1,p_regime_2,p_regime_3,p_regime_4
0,2022-08-14 20:00:00+00:00,1.318347e-13,3.609498e-06,3.520805e-07,9.999581e-01,3.792597e-05
1,2022-08-15 00:00:00+00:00,1.217233e-13,6.551903e-08,8.569524e-09,9.973258e-01,2.674132e-03
2,2022-08-15 04:00:00+00:00,1.608136e-07,4.437105e-05,1.337175e-02,9.861068e-01,4.769189e-04
3,2022-08-15 08:00:00+00:00,5.058652e-11,2.572369e-07,2.642392e-04,9.997353e-01,2.340134e-07
4,2022-08-15 12:00:00+00:00,4.998619e-12,3.278228e-09,1.669312e-05,9.999828e-01,5.006427e-07
5,2022-08-15 16:00:00+00:00,3.363817e-11,3.429047e-10,2.397807e-04,9.997600e-01,1.778424e-07
6,2022-08-15 20:00:00+00:00,9.254963e-13,1.558940e-09,1.394811e-05,9.999856e-01,4.150307e-07
7,2022-08-16 00:00:00+00:00,9.330820e-12,2.993546e-10,1.084406e-04,9.998910e-01,5.501113e-07
8,2022-08-16 04:00:00+00:00,1.155054e-12,6.143474e-10,2.557699e-05,9.999742e-01,2.151424e-07
9,2022-08-16 08:00:00+00:00,3.035835e-13,1.041008e-10,8.390559e-06,9.999913e-01,2.670755e-07


In [29]:
hmm_probs[["p_regime_0", "p_regime_1", "p_regime_2"]].sum(axis=1).describe()

count    1.026000e+04
mean     1.000000e+00
std      6.882021e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [78]:
h = 24

res = preds_hmm.copy()
res["regime"] = res["Y_pred"].apply(lambda p: int(np.argmax(p)))
res = res.reset_index().rename(columns={"index": "datetime"})

px = df[["datetime", "close", "r"]].copy()
px["datetime"] = pd.to_datetime(px["datetime"], utc=True)
px = px.sort_values("datetime")

px["fwd_ret"] = px["close"].shift(-h) / px["close"] - 1
px["fwd_log_ret"] = px["r"].rolling(h).sum().shift(-h + 1)

check = res.merge(
    px[["datetime", "r", "fwd_ret", "fwd_log_ret"]],
    on="datetime",
    how="inner",
)

print(
    check.groupby("regime")[["r", "fwd_ret", "fwd_log_ret"]]
         .agg(["count", "mean", "median", "std"])
)

           r                               fwd_ret                      \
       count      mean    median       std   count      mean    median   
regime                                                                   
0       2631 -0.000377  0.000023  0.011836    2631  0.003761  0.001727   
1       1592 -0.000281  0.000026  0.011041    1592  0.007766  0.002747   
2       2542 -0.000041  0.000172  0.010293    2542  0.003517  0.003338   
3       1965  0.000040  0.000192  0.010179    1965 -0.007994 -0.005891   
4       1350  0.001455  0.000329  0.011655    1350  0.008302  0.006166   

                 fwd_log_ret                                
             std       count      mean    median       std  
regime                                                      
0       0.054729        2631  0.001813  0.001445  0.054852  
1       0.055129        1592  0.005639  0.002476  0.054556  
2       0.057971        2542  0.001902  0.003449  0.058490  
3       0.051508        1965 -0.009092 -0